In [ ]:
# Cell 1: Imports and Setup
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor
from sklearn.decomposition import PCA

%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Cell 2: Segmentation Function
def segment_image(image_path, device="cuda"):
    """
    Hybrid segmentation: instance + panoptic.
    Returns segments with masks and bboxes.
    """
    EXCLUDED_LABELS = {
        'wall', 'floor', 'ceiling', 'window', 'door', 'sky', 'ground',
        'wall-other-merged', 'building'
    }
    
    img = Image.open(image_path).convert("RGB")
    img_width, img_height = img.size
    img_np = np.array(img)
    
    segments = []
    next_id = 0
    
    # === Instance Segmentation ===
    print("Running instance segmentation...")
    model_name = "facebook/mask2former-swin-large-coco-instance"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        instance_result = processor.post_process_instance_segmentation(
            outputs, 
            target_sizes=[img.size[::-1]],
            threshold=0.45
        )[0]
    
    segmentation_map = instance_result["segmentation"]
    segments_info = instance_result["segments_info"]
    
    for segment in segments_info:
        segment_id = segment['id']
        label_id = segment['label_id']
        score = segment.get('score', 1.0)
        segment_label = model.config.id2label[label_id].lower()
        
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        mask_tensor = (segmentation_map == segment_id)
        mask_np = mask_tensor.cpu().numpy()
        
        coords = np.argwhere(mask_np)
        if len(coords) == 0 or mask_np.sum() < 2000:
            continue
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        segments.append({
            'id': next_id,
            'label': model.config.id2label[label_id],
            'bbox': {
                'x_min': int(x_min),
                'y_min': int(y_min),
                'width': int(width),
                'height': int(height),
            },
            'mask': mask_np,
            'confidence': float(score),
            'source': 'instance'
        })
        next_id += 1
    
    print(f"  Found {len(segments)} instances")
    
    # === Panoptic Segmentation ===
    print("Running panoptic segmentation...")
    model_name = "facebook/mask2former-swin-large-coco-panoptic"
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(model_name).to(device)
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model(**inputs)
        panoptic_result = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[img.size[::-1]]
        )[0]
    
    segmentation_map = panoptic_result["segmentation"]
    segments_info = panoptic_result["segments_info"]
    
    STUFF_CLASSES = {
        'cabinet', 'light', 'shelf', 'counter', 'countertop',
        'table', 'desk', 'dining table', 'coffee table'
    }
    
    for segment in segments_info:
        segment_id = segment['id']
        label_id = segment['label_id']
        segment_label = model.config.id2label[label_id].lower()
        
        if not any(stuff in segment_label for stuff in STUFF_CLASSES):
            continue
        
        if any(excluded in segment_label for excluded in EXCLUDED_LABELS):
            continue
        
        mask_tensor = (segmentation_map == segment_id)
        mask_np = mask_tensor.cpu().numpy()
        
        coords = np.argwhere(mask_np)
        if len(coords) == 0:
            continue
        
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        width = x_max - x_min + 1
        height = y_max - y_min + 1
        
        segments.append({
            'id': next_id,
            'label': model.config.id2label[label_id],
            'bbox': {
                'x_min': int(x_min),
                'y_min': int(y_min),
                'width': int(width),
                'height': int(height),
            },
            'mask': mask_np,
            'confidence': 0.9,
            'source': 'panoptic'
        })
        next_id += 1
    
    print(f"  Total objects: {len(segments)}\n")
    
    return segments, img_width, img_height, img_np

In [ ]:
def estimate_depth(image_path, device="cuda", smooth=True):
    """
    Estimate depth map using MiDaS with optional smoothing.
    Returns normalized depth map (closer = smaller values).
    """
    print("Estimating depth with MiDaS...")
    
    # Load MiDaS
    midas = torch.hub.load("intel-isl/MiDaS", "DPT_Large")
    midas.to(device)
    midas.eval()
    
    midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
    transform = midas_transforms.dpt_transform
    
    # Load image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Inference
    input_batch = transform(img_rgb).to(device)
    
    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    
    depth_map = prediction.cpu().numpy()
    
    # Optional smoothing to reduce noise on flat surfaces
    if smooth:
        depth_map = cv2.bilateralFilter(depth_map.astype(np.float32), d=9, sigmaColor=75, sigmaSpace=75)
    
    # Normalize to [0, 1] range
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
    
    print(f"  Depth map shape: {depth_map.shape}")
    print(f"  Depth range: [{depth_map.min():.3f}, {depth_map.max():.3f}]")
    print(f"  Smoothing: {'enabled' if smooth else 'disabled'}\n")
    
    return depth_map

In [ ]:
# Cell 4: 3D Unprojection Function
def unproject_to_3d(pixel_coords, depth_values, img_width, img_height, fov_deg=60):
    """
    Convert 2D pixel coordinates + depth to 3D world coordinates.
    
    Args:
        pixel_coords: Nx2 array of (row, col) pixel coordinates
        depth_values: N array of depth values at those pixels
        img_width, img_height: Image dimensions
        fov_deg: Camera field of view in degrees
    
    Returns:
        Nx3 array of (X, Y, Z) world coordinates in meters
    """
    # Camera intrinsics (simple pinhole model)
    focal_length = img_width / (2 * np.tan(np.radians(fov_deg) / 2))
    cx = img_width / 2
    cy = img_height / 2
    
    # Extract pixel coordinates
    rows = pixel_coords[:, 0]  # y in image
    cols = pixel_coords[:, 1]  # x in image
    
    # Convert depth from [0,1] normalized to approximate meters
    # Assuming depth=0 means close, depth=1 means far
    # Scale to reasonable room dimensions (e.g., 0.5m to 10m)
    Z = 0.5 + depth_values * 9.5
    
    # Unproject to 3D
    X = (cols - cx) * Z / focal_length
    Y = (rows - cy) * Z / focal_length
    
    # Stack into Nx3 array
    points_3d = np.column_stack([X, Y, Z])
    
    return points_3d

In [ ]:
# Cell 4.5: Edge Detection Helper
def extract_surface_points(mask, method='contour'):
    """
    Extract surface/edge points from a binary mask.
    
    Args:
        mask: Binary mask (H x W)
        method: 'contour' (outer boundary) or 'canny' (all edges)
    
    Returns:
        Nx2 array of (row, col) coordinates of surface points
    """
    if method == 'contour':
        # Find the outer contour of the mask
        mask_uint8 = mask.astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        
        if not contours:
            return np.argwhere(mask)  # Fallback to all points
        
        # Get the largest contour
        largest_contour = max(contours, key=cv2.contourArea)
        
        # Extract coordinates
        surface_points = largest_contour.squeeze()  # Shape: (N, 2) -> (col, row)
        
        if surface_points.ndim == 1:  # Handle case where contour has only 1 point
            surface_points = surface_points.reshape(1, -1)
        
        # Convert from (col, row) to (row, col)
        surface_points = surface_points[:, [1, 0]]
        
        return surface_points
    
    elif method == 'canny':
        # Use Canny edge detection
        mask_uint8 = mask.astype(np.uint8) * 255
        edges = cv2.Canny(mask_uint8, threshold1=50, threshold2=150)
        
        edge_points = np.argwhere(edges > 0)  # (row, col)
        
        if len(edge_points) == 0:
            return np.argwhere(mask)  # Fallback
        
        return edge_points
    
    else:
        raise ValueError(f"Unknown method: {method}")

In [ ]:
def estimate_3d_orientations(segments, depth_map, img_width, img_height, reference_id=None, 
                            downsample=True, use_surface_only=True):
    """
    Estimate 3D orientations using PCA on unprojected 3D points.
    Returns orientation vectors in the horizontal (X-Z) plane.
    
    Args:
        use_surface_only: If True, use only edge/boundary points for PCA
    """
    print("\n" + "="*80)
    print(f"3D ORIENTATION ESTIMATION (Depth-Aware PCA)")
    if use_surface_only:
        print("Using SURFACE-ONLY mode (edges/contours)")
    print("="*80 + "\n")
    
    orientation_results = []
    
    for seg in segments:
        # Get 2D pixel coordinates
        if use_surface_only:
            # Extract surface/edge points only
            coords_2d = extract_surface_points(seg['mask'], method='contour')
            print(f"{seg['label']} (ID: {seg['id']}): Using {len(coords_2d)} surface points")
        else:
            # Use all mask points
            coords_2d = np.argwhere(seg['mask'])  # Shape: (N, 2) -> (row, col)
        
        if len(coords_2d) < 10:
            orientation_results.append({
                'id': seg['id'],
                'label': seg['label'],
                'orientation_3d': None,
                'orientation_3d_pc2': None,
                'confidence': 0.0,
                'pc1_vertical': False,
                'bbox_area': seg['bbox']['width'] * seg['bbox']['height'],
                'warning': 'Too few points'
            })
            continue
        
        # Downsample points to reduce noise (every Nth point)
        if downsample and len(coords_2d) > 500:
            step = max(1, len(coords_2d) // 500)
            coords_2d = coords_2d[::step]
        
        # Get depth values at these pixel locations
        depth_values = depth_map[coords_2d[:, 0], coords_2d[:, 1]]
        
        # Unproject to 3D world coordinates
        points_3d = unproject_to_3d(coords_2d, depth_values, img_width, img_height)
        
        # Extract horizontal (X, Z) coordinates only - ignore vertical component entirely
        points_horizontal = points_3d[:, [0, 2]]  # Only X and Z columns
                
        # Run PCA on purely horizontal data
        pca_2d = PCA(n_components=2)
        pca_2d.fit(points_horizontal)

        # Get principal components in horizontal plane
        pc1_horizontal = pca_2d.components_[0]  # 2D vector: [X, Z]
        pc2_horizontal = pca_2d.components_[1]  # 2D vector: [X, Z]
        confidence = pca_2d.explained_variance_ratio_[0]

        # Convert to 3D format for consistency (but Y is always 0)
        orientation_xz_pc1_norm = np.array([pc1_horizontal[0], 0, pc1_horizontal[1]])
        orientation_xz_pc2_norm = np.array([pc2_horizontal[0], 0, pc2_horizontal[1]])

        # For logging, create "full 3D" versions (but Y component is forced to 0)
        pc1_3d = orientation_xz_pc1_norm
        pc2_3d = orientation_xz_pc2_norm

        # No need to check vertical dominance since we're ignoring Y entirely
        is_vertical = False
        y_dominance = 0.0
        
        warning = None
        
        orientation_results.append({
            'id': seg['id'],
            'label': seg['label'],
            'orientation_3d': orientation_xz_pc1_norm,  # Normalized (X, 0, Z) from PC1
            'orientation_3d_pc2': orientation_xz_pc2_norm,  # Alternative from PC2
            'pc1_full': pc1_3d,  # Full 3D vector for reference
            'pc2_full': pc2_3d,  # Second principal component
            'confidence': confidence,
            'pc1_vertical': is_vertical,
            'y_dominance': y_dominance,
            'bbox_area': seg['bbox']['width'] * seg['bbox']['height'],
            'warning': warning
        })
        
        print(f"  PC1 (3D): [{pc1_3d[0]:6.3f}, {pc1_3d[1]:6.3f}, {pc1_3d[2]:6.3f}]")
        print(f"  PC2 (3D): [{pc2_3d[0]:6.3f}, {pc2_3d[1]:6.3f}, {pc2_3d[2]:6.3f}]")
        print(f"  Confidence: {confidence:.3f}\n")
    
    # Select reference object (prefer high confidence, non-vertical, large objects)
    if reference_id is None:
        candidates = [r for r in orientation_results 
                     if r['orientation_3d'] is not None 
                     and r['confidence'] > 0.3 
                     and not r['pc1_vertical']]
        
        if candidates:
            # Prefer large, confident, non-vertical objects
            reference = max(candidates, key=lambda x: x['confidence'] * x['bbox_area'])
        else:
            # Fallback to any valid object
            candidates = [r for r in orientation_results if r['orientation_3d'] is not None]
            reference = max(candidates, key=lambda x: x['bbox_area']) if candidates else orientation_results[0]
        
        reference_id = reference['id']
    else:
        reference = next(r for r in orientation_results if r['id'] == reference_id)
    
    print("="*80)
    print(f"REFERENCE: {reference['label']} (ID: {reference_id})")
    print("="*80 + "\n")
    
    return orientation_results, reference_id

In [ ]:
def compute_relative_angles(orientation_results, reference_id, use_pc2_for=None):
    """
    Compute angles between each object's orientation and the reference.
    
    Args:
        orientation_results: Results from estimate_3d_orientations
        reference_id: ID of reference object
        use_pc2_for: Set of labels that should use PC2 instead of PC1 (e.g., {'tvmonitor'})
    """
    if use_pc2_for is None:
        use_pc2_for = set()
    
    reference = next(r for r in orientation_results if r['id'] == reference_id)
    ref_orientation = reference['orientation_3d']
    
    print("RELATIVE ANGLES TO REFERENCE")
    print("="*80 + "\n")
    
    angle_results = []
    
    for result in orientation_results:
        if result['orientation_3d'] is None:
            angle_results.append({
                **result,
                'angle_to_reference': None,
                'angle_to_reference_pc2': None,
                'is_reference': False,
                'relationship': 'UNKNOWN',
                'used_pc2': False
            })
            continue
        
        is_reference = (result['id'] == reference_id)
        
        # Decide which orientation to use
        use_pc2 = any(label.lower() in result['label'].lower() for label in use_pc2_for)
        orientation = result['orientation_3d_pc2'] if use_pc2 else result['orientation_3d']
        
        if is_reference:
            angle_between = 0.0
            angle_between_pc2 = 0.0
            relationship = "REFERENCE"
        else:
            # Compute angle using PC1
            dot_product = np.dot(result['orientation_3d'], ref_orientation)
            dot_product = np.clip(dot_product, -1.0, 1.0)
            angle_between = np.degrees(np.arccos(np.abs(dot_product)))
            
            # Compute angle using PC2 (alternative)
            dot_product_pc2 = np.dot(result['orientation_3d_pc2'], ref_orientation)
            dot_product_pc2 = np.clip(dot_product_pc2, -1.0, 1.0)
            angle_between_pc2 = np.degrees(np.arccos(np.abs(dot_product_pc2)))
            
            # Use the appropriate angle
            angle_to_use = angle_between_pc2 if use_pc2 else angle_between
            
            # Classify relationship
            if result['pc1_vertical']:
                relationship = "VERTICAL (unreliable)"
            elif angle_to_use < 15:
                relationship = "PARALLEL"
            elif 75 < angle_to_use < 105:
                relationship = "PERPENDICULAR"
            else:
                relationship = f"ANGLED"
        
        angle_results.append({
            **result,
            'angle_to_reference': angle_between,
            'angle_to_reference_pc2': angle_between_pc2,
            'is_reference': is_reference,
            'relationship': relationship,
            'used_pc2': use_pc2
        })
        
        if is_reference:
            print(f"✓ {result['label']:20s} (ID: {result['id']}): 0.0° [REFERENCE]")
        else:
            pc_indicator = " [using PC2]" if use_pc2 else ""
            angle_display = angle_between_pc2 if use_pc2 else angle_between
            print(f"  {result['label']:20s} (ID: {result['id']}): {angle_display:5.1f}° - {relationship}{pc_indicator}")
            if not use_pc2 and abs(angle_between - angle_between_pc2) > 30:
                print(f"    └─ PC2 alternative: {angle_between_pc2:5.1f}°")
    
    print()
    return angle_results

In [ ]:
# Cell 7: Visualization Function
def visualize_3d_orientations(segments, angle_results, img_np, depth_map):
    """
    Visualize objects with their 3D orientation vectors.
    """
    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    axes = axes.flatten()
    
    for idx, seg in enumerate(segments[:6]):
        if idx >= 6:
            break
        
        ax = axes[idx]
        result = next(r for r in angle_results if r['id'] == seg['id'])
        
        bbox = seg['bbox']
        x_min, y_min = bbox['x_min'], bbox['y_min']
        x_max = x_min + bbox['width']
        y_max = y_min + bbox['height']
        
        pad = 20
        x_min = max(0, x_min - pad)
        y_min = max(0, y_min - pad)
        x_max = min(img_np.shape[1], x_max + pad)
        y_max = min(img_np.shape[0], y_max + pad)
        
        cropped_img = img_np[y_min:y_max, x_min:x_max]
        cropped_mask = seg['mask'][y_min:y_max, x_min:x_max]
        
        # Create overlay
        overlay = cropped_img.copy()
        overlay[cropped_mask] = [0, 255, 0]
        blended = cv2.addWeighted(cropped_img, 0.6, overlay, 0.4, 0)
        
        ax.imshow(blended)
        
        # Draw orientation arrow
        center_y, center_x = cropped_mask.shape[0] // 2, cropped_mask.shape[1] // 2
        arrow_length = min(cropped_mask.shape) // 3
        
        if result['orientation_3d'] is not None:
            # Project 3D orientation back to image space for visualization
            # Use X and Z components (horizontal orientation)
            orientation = result['orientation_3d']
            
            # Map to image: X → horizontal, Z → "depth" (we'll show as horizontal too)
            # This is approximate visualization - showing the horizontal component
            dx = arrow_length * orientation[0]  # X component
            dy = arrow_length * orientation[2]  # Z component (depth)
            
            color = 'blue' if result['is_reference'] else 'red'
            ax.arrow(center_x, center_y, dx, -dy,  # Negative dy because image y is inverted
                    color=color, width=3, head_width=15, head_length=10)
        
        # Title with relationship
        title = f"{seg['label']} (ID: {seg['id']})\n"
        if result['is_reference']:
            title += "REFERENCE"
        elif result['angle_to_reference'] is not None:
            title += f"{result['relationship']}: {result['angle_to_reference']:.1f}°"
        else:
            title += "UNKNOWN"
        title += f"\n(conf: {result['confidence']:.2f})"
        
        ax.set_title(title, fontsize=10, weight='bold')
        ax.axis('off')
    
    for idx in range(len(segments), 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('3d_orientations.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Also show depth map
    plt.figure(figsize=(12, 6))
    plt.imshow(depth_map, cmap='viridis')
    plt.colorbar(label='Depth (normalized)')
    plt.title('Depth Map')
    plt.axis('off')
    plt.savefig('depth_map.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 7.5: Visualize 3D Point Clouds
def visualize_3d_point_clouds(segments, depth_map, img_width, img_height, orientation_results, max_objects=6):
    """
    Visualize the actual 3D point clouds with PCA vectors in true 3D space.
    """
    from mpl_toolkits.mplot3d import Axes3D
    
    fig = plt.figure(figsize=(20, 12))
    
    num_plots = min(len(segments), max_objects)
    
    for idx, seg in enumerate(segments[:num_plots]):
        ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
        
        # Get 2D coordinates
        coords_2d = np.argwhere(seg['mask'])
        
        if len(coords_2d) < 10:
            continue
        
        # Downsample for visualization
        if len(coords_2d) > 1000:
            step = len(coords_2d) // 1000
            coords_2d = coords_2d[::step]
        
        # Get depth and unproject
        depth_values = depth_map[coords_2d[:, 0], coords_2d[:, 1]]
        points_3d = unproject_to_3d(coords_2d, depth_values, img_width, img_height)
        
        # Plot point cloud
        ax.scatter(points_3d[:, 0], points_3d[:, 2], points_3d[:, 1], 
                  c='lightblue', alpha=0.3, s=1)
        
        # Get orientation vectors
        result = next(r for r in orientation_results if r['id'] == seg['id'])
        
        if result['orientation_3d'] is not None:
            # Calculate centroid
            centroid = points_3d.mean(axis=0)
            
            # Draw PC1 vector (scaled for visibility)
            pc1 = result['orientation_3d']
            scale = np.ptp(points_3d, axis=0).max() * 0.3  # Scale arrow
            
            ax.quiver(centroid[0], centroid[2], centroid[1],
                     pc1[0] * scale, pc1[2] * scale, pc1[1] * scale,
                     color='red', arrow_length_ratio=0.3, linewidth=3,
                     label='PC1')
            
            # Draw PC2 vector
            pc2 = result['orientation_3d_pc2']
            ax.quiver(centroid[0], centroid[2], centroid[1],
                     pc2[0] * scale, pc2[2] * scale, pc2[1] * scale,
                     color='blue', arrow_length_ratio=0.3, linewidth=3,
                     label='PC2')
        
        # Set labels
        ax.set_xlabel('X (left-right)', fontsize=8)
        ax.set_ylabel('Z (depth)', fontsize=8)
        ax.set_zlabel('Y (up-down)', fontsize=8)
        ax.set_title(f"{seg['label']} (ID: {seg['id']})\nPC1: [{pc1[0]:.2f}, {pc1[1]:.2f}, {pc1[2]:.2f}]",
                    fontsize=10)
        
        # Set equal aspect ratio
        max_range = np.array([
            points_3d[:, 0].max() - points_3d[:, 0].min(),
            points_3d[:, 2].max() - points_3d[:, 2].min(),
            points_3d[:, 1].max() - points_3d[:, 1].min()
        ]).max() / 2.0
        
        mid_x = (points_3d[:, 0].max() + points_3d[:, 0].min()) * 0.5
        mid_z = (points_3d[:, 2].max() + points_3d[:, 2].min()) * 0.5
        mid_y = (points_3d[:, 1].max() + points_3d[:, 1].min()) * 0.5
        
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_z - max_range, mid_z + max_range)
        ax.set_zlim(mid_y - max_range, mid_y + max_range)
        
        # Add legend
        ax.legend(loc='upper right', fontsize=8)
        
        # Set viewing angle
        ax.view_init(elev=20, azim=45)
    
    plt.tight_layout()
    plt.savefig('3d_point_clouds.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 8: Run Full Pipeline
IMAGE_PATH = "input/kitchen.jpg"  # Replace with your image path

# Step 1: Segment image
segments, img_w, img_h, img_np = segment_image(IMAGE_PATH, DEVICE)

# Step 2: Estimate depth
depth_map = estimate_depth(IMAGE_PATH, DEVICE, smooth=True)

# Step 3: Estimate 3D orientations using depth-aware PCA
orientation_results, reference_id = estimate_3d_orientations(
    segments, depth_map, img_w, img_h, use_surface_only=False  # NEW: Enable surface-only mode
)

visualize_3d_point_clouds(segments, depth_map, img_w, img_h, orientation_results)

# Step 4: Compute relative angles
angle_results = compute_relative_angles(
    orientation_results, 
    reference_id,
)
# Step 5: Visualize
visualize_3d_orientations(segments, angle_results, img_np, depth_map)